# 1 · The data, and why one agent is not enough

276 accounts were flagged over a weekend by eight automated rules. Roughly two
thirds of them did nothing wrong.

This notebook establishes three things before any agent code exists:

1. what is actually in the database
2. that **no rule is reliable**, so the queue cannot be triaged by which rule fired
3. two properties of the data that quietly break the obvious query

In [1]:
# Reload the package from disk on every run, so an edit to src/sentinel takes
# effect without restarting the kernel. Python caches imported modules in
# sys.modules and a stale one will happily report yesterday's numbers.
import sys, pathlib
for name in [m for m in sys.modules if m.startswith("sentinel")]:
    del sys.modules[name]

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("sentinel package:", ROOT / "src" / "sentinel")

sentinel package: D:\AgentBuilder2026datasense\Sentinel-MultiagentFraudAnalyst\src\sentinel


In [2]:
from sentinel import db, queries

with db.read_only() as conn:
    for table in ["alerts", "rules", "transactions", "customers", "accounts",
                  "cards", "devices", "case_notes", "disputes", "prior_cases", "merchants"]:
        n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"{table:16} {n:>8,}")

print()
print("alerted accounts:", len(queries.alerted_accounts()))

alerts                411
rules                   8
transactions      108,249
customers           1,200
accounts            1,200
cards               1,458
devices             1,520
case_notes            260
disputes               86
prior_cases           200
merchants             400

alerted accounts: 276


## The database is opened read-only, structurally

`mode=ro` refuses at the file level and `PRAGMA query_only` refuses at the
statement level. This is not a filter that scans SQL for the word INSERT — there
is no write path to slip past.

In [3]:
import sqlite3
try:
    with db.read_only() as conn:
        conn.execute("INSERT INTO alerts (alert_id) VALUES ('X')")
    print("FAILED - the database accepted a write")
except sqlite3.OperationalError as exc:
    print("refused:", exc)

refused: attempt to write a readonly database


## No rule is reliable

Every rule fires on both fraud and legitimate customers. The two that fire most
are the two least reliable, so "which rule fired" carries almost no signal.

In [4]:
rows = db.query('''
    SELECT r.rule_id, r.name, r.description, COUNT(*) AS fired
    FROM alerts a JOIN rules r USING(rule_id)
    GROUP BY r.rule_id ORDER BY fired DESC
''')
for r in rows:
    print(f"{r['rule_id']}  {r['name']:<28} fired {r['fired']:>3}")
    print(f"      {r['description']}")

R02  New device high value        fired  88
      Transaction above 25,000 from a device first seen in the last 24 hours.
R03  Impossible travel            fired  83
      Two authorisations from different countries less than 3 hours apart.
R01  Velocity spike               fired  66
      More than 6 authorisations on one card within 60 minutes.
R04  Card testing pattern         fired  49
      Five or more authorisations under 100 within 30 minutes.
R08  Limit approach               fired  47
      Cumulative spend crosses 90 percent of credit limit within 48 hours.
R05  High risk merchant burst     fired  41
      Three or more transactions at crypto, gift card or money transfer merchants in 24 hours.
R07  Night time high value        fired  37
      Transaction above 40,000 between 01:00 and 05:00 local.


## Quirk 1 · `triggered_at` is not when the offending transaction happened

The obvious query is "give me the transactions in the hours before the alert
fired". On this data that is wrong most of the time.

In [5]:
r = db.query_one('''
    SELECT SUM(CASE WHEN t.ts > a.triggered_at THEN 1 ELSE 0 END) AS after,
           SUM(CASE WHEN t.ts <= a.triggered_at THEN 1 ELSE 0 END) AS before,
           COUNT(*) AS total,
           MAX(ROUND((julianday(t.ts) - julianday(a.triggered_at)) * 24, 1)) AS max_hours
    FROM alerts a JOIN transactions t ON t.txn_id = a.trigger_txn_id
''')
print(f"trigger transaction lands AFTER the alert : {r['after']} of {r['total']}")
print(f"                          ... by up to    : {r['max_hours']} hours")

trigger transaction lands AFTER the alert : 342 of 411
                          ... by up to    : 18.0 hours


In [6]:
# What that costs you, on one account.
acct = "A00985"
lo, hi = queries.incident_window(acct)
episode = queries.get_incident_activity(acct)

naive = db.query('''
    SELECT COUNT(*) n, ROUND(SUM(amount)) total FROM transactions
    WHERE account_id = ?
      AND ts BETWEEN strftime('%Y-%m-%dT%H:%M:%S',
                              (SELECT MIN(triggered_at) FROM alerts WHERE account_id = ?), '-6 hours')
                 AND (SELECT MIN(triggered_at) FROM alerts WHERE account_id = ?)
''', (acct, acct, acct))[0]

print(f"looking backwards from triggered_at : {naive['n']} txns, {naive['total']:,.0f}")
print(f"incident_window()                   : {len(episode)} txns, "
      f"{sum(t['amount'] for t in episode):,.0f}")

looking backwards from triggered_at : 1 txns, 36,861
incident_window()                   : 4 txns, 216,091


## Quirk 2 · A note's *timing* decides what it means

The same sentence means opposite things depending on when it was written.

- filed **before** the incident: a pre-existing explanation, and often decisive
- filed **after**: the customer reacting. *"I did not make these transactions"*
  corroborates fraud rather than excusing it

Models are poor at date arithmetic, so this is computed in SQL and handed over
as a label.

In [7]:
r = db.query_one('''
    WITH first_alert AS (SELECT account_id, MIN(triggered_at) AS at FROM alerts GROUP BY account_id)
    SELECT SUM(CASE WHEN n.created_at <  f.at THEN 1 ELSE 0 END) AS before_incident,
           SUM(CASE WHEN n.created_at >= f.at THEN 1 ELSE 0 END) AS after_incident
    FROM case_notes n
    JOIN accounts ac ON ac.customer_id = n.customer_id
    JOIN first_alert f ON f.account_id = ac.account_id
''')
print("notes filed before the incident:", r["before_incident"])
print("notes filed after  the incident:", r["after_incident"])

notes filed before the incident: 179
notes filed after  the incident: 71


## The case that makes the argument

`A00985`. The numbers say account takeover. One note, filed five hours earlier,
explains the whole thing. No threshold on the numbers can find that.

In [8]:
for a in queries.get_alerts(acct):
    print(f"{a['alert_id']}  {a['rule_id']} {a['rule_name']}  fired {a['triggered_at']}")
print()
for n in queries.get_case_notes(acct):
    print(f"{n['note_id']}  {n['created_at']}  [{n['timing']}]")
    print(f'  "{n["note"]}"')

AL0170  R02 New device high value  fired 2026-02-27T12:46:44

N00080  2026-02-27T07:46:44  [before_incident]
  "Support chat. Customer upgraded their phone on the 14th and could not log in. Walked them through re-registration. Verified with video KYC."
